# Three-Way FRET Simulation

Simulates one-donor / two-acceptor FRET (Hohng, Joo & Ha, *Biophys. J.* **87**, 1328–1337, 2004;
doi:10.1529/biophysj.104.043935) and predicts the Bayer-camera RGB colour ratios as a function
of inter-dye distances, using the Nile Red / S. *aureus* filter set (515 nm longpass dichroic).

Förster radii are computed from fpbase spectra following Clegg, *Methods Enzymol.* **211**, 353–388 (1992);
doi:10.1016/0076-6879(92)11020-J.

## Physical model

For donor D competing between two acceptors A1 and A2:

$$E_1 = \frac{(r_1/R_{0,DA1})^{-6}}{1 + (r_1/R_{0,DA1})^{-6} + (r_2/R_{0,DA2})^{-6}}, \quad E_2 = \frac{(r_2/R_{0,DA2})^{-6}}{1 + (r_1/R_{0,DA1})^{-6} + (r_2/R_{0,DA2})^{-6}}$$

Photon budget (I₀ = unquenched donor photons, κ² = 2/3, n = 1.33):

$$N_D = I_0(1-E_1-E_2), \quad N_{A1} = I_0 \frac{E_1}{\Phi_D}\Phi_{A1}, \quad N_{A2} = I_0 \frac{E_2}{\Phi_D}\Phi_{A2}$$

**Note:** A1→A2 energy transfer is *not* modelled; R₀(A1,A2) is computed and flagged.

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import fpbase
import mpltern

sys.path.append('../..')
import src.SpectralFunctions as SpectralFunctions

# --- camera setup ---
sf = SpectralFunctions.Spectral_Funcs()
R, G, B, wavelength = sf.getpixelefficiency()
pixel_QYs = np.vstack([B, G, R])   # shape (3, n_wl); columns → [B, G, R]

# --- filter set: 515 nm longpass (Nile Red / S. aureus configuration) ---
filters = [
    'semrock-di03-r514-t1-25x36',
    'semrock-ff01-515-lp',
    'nikon-ti2-non-multiphoton-pfs-dichroic',
]

# --- photon budget ---
I0 = 10_000   # unquenched donor photons (high-SNR condition)
N_MIN = 500   # minimum acceptor photons before shot-noise dominates

# --- physical constants ---
KAPPA_SQ = 2 / 3
N_REFR   = 1.33

## Helper functions

In [ ]:
from matplotlib.ticker import MultipleLocator
import mpltern  # noqa: ensure mpltern is loaded before any ternary axes


def _ternary_axis_setup(ax):
    """Apply the same axis style as PlottingFunctions.Plotter._setup_ternary_axis.

    Convention (matching PlottingFunctions.ternary_scatter_plot):
        ax.scatter(R, G, B)  →  top = R,  left = G,  right = B
    """
    for axis in [ax.taxis, ax.laxis, ax.raxis]:
        axis.set_major_locator(MultipleLocator(0.2))
        axis.set_minor_locator(MultipleLocator(0.1))
    ax.set_ternary_lim(0, 1, 0, 1, 0, 1)
    ax.grid(lw=0.5, alpha=0.25, ls='--', which='both', axis='both', color='gray')


def get_FRET_pair(donor_dye, acceptor_dye):
    """Fetch emission (donor) and absorption (acceptor) spectra from fpbase.

    For the acceptor, prefers 'AB' (molar extinction coefficient); falls back
    to 'EX' (normalised excitation spectrum) if AB is unavailable.
    When EX is used, the spectrum is scaled to ext_coeff at its peak so that
    the overlap integral J has consistent units.
    """
    donor_obj  = fpbase.get_fluorophore(donor_dye).default_state
    em_data    = np.array([x for x in donor_obj.spectra if 'EM' in x.subtype][0].data)
    donor      = pd.DataFrame({'wavelength': em_data[:, 0],
                                'Emission (AU)': np.clip(em_data[:, 1], 0, None)})
    QY_D       = donor_obj.qy or 0.01

    acceptor_obj = fpbase.get_fluorophore(acceptor_dye).default_state
    QY_A         = acceptor_obj.qy or 0.01

    ab_matches = [x for x in acceptor_obj.spectra if 'AB' in x.subtype]
    ex_matches = [x for x in acceptor_obj.spectra if 'EX' in x.subtype]

    if ab_matches:
        ab_data = np.array(ab_matches[0].data)
        ab_data[:, 1] = np.clip(ab_data[:, 1], 0, None) * acceptor_obj.ext_coeff
    elif ex_matches:
        ab_data = np.array(ex_matches[0].data)
        ab_data[:, 1] = np.clip(ab_data[:, 1], 0, None)
        mx = ab_data[:, 1].max()
        if mx > 0:
            ab_data[:, 1] = ab_data[:, 1] / mx * acceptor_obj.ext_coeff
    else:
        raise ValueError(f"No AB or EX spectrum found for acceptor '{acceptor_dye}' in fpbase")

    acceptor = pd.DataFrame({'wavelength': ab_data[:, 0], 'absorption': ab_data[:, 1]})
    return acceptor, donor, QY_D, QY_A


def forster_radius_nm(donor_name, acceptor_name, n=N_REFR, kappa_sq=KAPPA_SQ):
    """Return (R0_nm, QY_D, QY_A) for a donor/acceptor pair.

    Spectral overlap integral J is computed over the wavelength range shared
    by donor emission and acceptor absorption, following Clegg (1992).
    """
    acceptor, donor, QY_D, QY_A = get_FRET_pair(donor_name, acceptor_name)

    wl_start = max(acceptor['wavelength'].iloc[0],  donor['wavelength'].iloc[0])
    wl_end   = min(acceptor['wavelength'].iloc[-1], donor['wavelength'].iloc[-1])
    donor    = donor[(donor['wavelength']    > wl_start) & (donor['wavelength']    < wl_end)].copy()
    acceptor = acceptor[(acceptor['wavelength'] > wl_start) & (acceptor['wavelength'] < wl_end)].copy()

    donor['Emission (norm)'] = donor['Emission (AU)'] / np.trapz(donor['Emission (AU)'])

    wl_m    = acceptor['wavelength'].to_numpy() * 1e-9
    overlap = acceptor['absorption'].to_numpy() * donor['Emission (norm)'].to_numpy()
    J       = np.trapz(overlap * wl_m ** 4)

    R0_m  = (2.11e-5) * (kappa_sq * n ** (-4) * QY_D * J) ** (1 / 6)
    R0_nm = R0_m * 1e9
    return R0_nm, QY_D, QY_A


def dye_rgb(dye_name):
    """Return normalised [B, G, R] colour vector for a single dye through the filter set.

    Index convention: rgb[0]=B, rgb[1]=G, rgb[2]=R
    (matches pixel_QYs row order: np.vstack([B, G, R]))
    """
    _, rgb = sf.get_pixel_fractions_dye_and_filters(
        [dye_name], filters, wavelength, pixel_QYs, normalized=True
    )
    rgb = np.asarray(rgb).ravel()
    return rgb / rgb.sum()


def triad_photons(r1, r2, R0_DA1, R0_DA2, QY_D, QY_A1, QY_A2, I0=I0):
    """Detected photon counts (N_D, N_A1, N_A2) for a triad at distances r1, r2 (nm)."""
    k1    = (r1 / R0_DA1) ** (-6)
    k2    = (r2 / R0_DA2) ** (-6)
    denom = 1 + k1 + k2
    E1    = k1 / denom
    E2    = k2 / denom
    N_D   = I0 * (1 - E1 - E2)
    N_A1  = I0 * (E1 / QY_D) * QY_A1
    N_A2  = I0 * (E2 / QY_D) * QY_A2
    return N_D, N_A1, N_A2


def triad_rgb_norm(r1, r2, R0_DA1, R0_DA2, QY_D, QY_A1, QY_A2,
                   rgb_D, rgb_A1, rgb_A2, I0=I0):
    """Normalised [B, G, R] colour vector and total photon count for a triad."""
    N_D, N_A1, N_A2 = triad_photons(r1, r2, R0_DA1, R0_DA2, QY_D, QY_A1, QY_A2, I0)
    rgb_obs = N_D * rgb_D + N_A1 * rgb_A1 + N_A2 * rgb_A2
    return rgb_obs / rgb_obs.sum(), rgb_obs.sum()


def triad_metrics(donor, acc1, acc2, I0=I0, N_min=N_MIN, crosstalk_threshold=3.0):
    """Compute all distinguishability metrics for one (donor, A1, A2) triad."""
    R0_DA1, QY_D, QY_A1 = forster_radius_nm(donor, acc1)
    R0_DA2, _,    QY_A2 = forster_radius_nm(donor, acc2)
    R0_A1A2, _,   _     = forster_radius_nm(acc1,  acc2)

    rgb_D  = dye_rgb(donor)
    rgb_A1 = dye_rgb(acc1)
    rgb_A2 = dye_rgb(acc2)

    r_inf  = 1000.0
    r_zero = 0.01
    P_D,  _ = triad_rgb_norm(r_inf,  r_inf,  R0_DA1, R0_DA2, QY_D, QY_A1, QY_A2, rgb_D, rgb_A1, rgb_A2, I0)
    P_A1, _ = triad_rgb_norm(r_zero, r_inf,  R0_DA1, R0_DA2, QY_D, QY_A1, QY_A2, rgb_D, rgb_A1, rgb_A2, I0)
    P_A2, _ = triad_rgb_norm(r_inf,  r_zero, R0_DA1, R0_DA2, QY_D, QY_A1, QY_A2, rgb_D, rgb_A1, rgb_A2, I0)

    v1   = P_A1[:2] - P_D[:2]
    v2   = P_A2[:2] - P_D[:2]
    area = 0.5 * abs(v1[0] * v2[1] - v1[1] * v2[0])

    min_dist = min(
        np.linalg.norm(P_D  - P_A1),
        np.linalg.norm(P_D  - P_A2),
        np.linalg.norm(P_A1 - P_A2),
    )

    _, N_A1_at_R0, _ = triad_photons(R0_DA1, r_inf, R0_DA1, R0_DA2, QY_D, QY_A1, QY_A2, I0)
    _, _, N_A2_at_R0 = triad_photons(r_inf, R0_DA2, R0_DA1, R0_DA2, QY_D, QY_A1, QY_A2, I0)

    return dict(
        donor=donor, acc1=acc1, acc2=acc2,
        R0_DA1=round(R0_DA1, 2), R0_DA2=round(R0_DA2, 2),
        QY_D=QY_D, QY_A1=QY_A1, QY_A2=QY_A2,
        N_A1_at_R0=round(N_A1_at_R0), N_A2_at_R0=round(N_A2_at_R0),
        area=round(area, 5), min_dist=round(min_dist, 4),
        R0_A1A2=round(R0_A1A2, 2),
        practical=bool(min(N_A1_at_R0, N_A2_at_R0) > N_min),
        A1A2_crosstalk=bool(R0_A1A2 > crosstalk_threshold),
        _P_D=P_D, _P_A1=P_A1, _P_A2=P_A2,
        _rgb_D=rgb_D, _rgb_A1=rgb_A1, _rgb_A2=rgb_A2,
        _R0_DA1=R0_DA1, _R0_DA2=R0_DA2,
        _QY_D=QY_D, _QY_A1=QY_A1, _QY_A2=QY_A2,
    )

In [ ]:
# Quick sanity check: print per-dye [B, G, R] fractions through the filter set.
# With a 515 nm longpass, red/NIR dyes should show small B, moderate G, large R.
# Blue dyes should show very small values overall (mostly blocked by LP filter).
_spot_check = ['Cy3', 'ATTO 647N', 'Cy5.5', 'ATTO 680']
print(f'{"Dye":15s}   {"B":>6s}  {"G":>6s}  {"R":>6s}  (sanity: red dyes → large R)')
print('-' * 45)
for d in _spot_check:
    try:
        rgb = dye_rgb(d)
        print(f'{d:15s}   {rgb[0]:.3f}   {rgb[1]:.3f}   {rgb[2]:.3f}')
    except Exception as e:
        print(f'{d:15s}   ERROR: {e}')

## Dye list → exhaustive triad generation

For every dye that has **at least two red-shifted dyes** in the list (i.e. dyes whose absorption peak is longer), it is treated as a potential donor and all C(n_acceptors, 2) acceptor pairs are generated. This is exhaustive — every valid donor is tried.

Direct excitation at 488, 515 and 561 nm is checked for every acceptor in every triad.

In [ ]:
from itertools import combinations

# ── user configuration ────────────────────────────────────────────────────────
dye_list = [
    'Cy3',
    'Cy3B',
    'ATTO 550',
    'ATTO 610',
    'ATTO 620',
    'ATTO 647N',
    'Cy5',
    'Cy5.5',
    'ATTO 594',
    'ATTO 532',
    'ATTO 542',
    'Alexa Fluor 594',
    'Alexa Fluor 546',
    'ATTO 520',
    'ATTO 514',
    'ATTO 680',
    'ATTO 700',
    'ATTO 725',
    'Alexa Fluor 700',
    'ATTO 740',
    'Alexa Fluor 680',
    # add / remove dyes here — names must match fpbase identifiers
]

LASER_WAVELENGTHS    = [515, 561]   # 488 nm line not available in this configuration

# A laser line is "viable for the donor" if the donor absorbs >= this fraction
# of its peak at that wavelength.
DONOR_EXC_THRESHOLD  = 0.40   # 40 %

# Per (triad, laser) row is laser_practical=False if either acceptor exceeds this
# fraction of its own peak absorption at that laser wavelength.
ACC_DIRECT_EXC_LIMIT = 0.15   # 15 %
# ─────────────────────────────────────────────────────────────────────────────


def _get_absorption_data(dye_name):
    """Return (wavelengths, normalised_values) for AB spectrum; falls back to EX."""
    obj = fpbase.get_fluorophore(dye_name).default_state
    for subtype in ('AB', 'EX'):
        matches = [x for x in obj.spectra if subtype in x.subtype]
        if matches:
            ab_data = np.array(matches[0].data)
            ab_data[:, 1] = np.clip(ab_data[:, 1], 0, None)
            if subtype == 'EX':
                mx = ab_data[:, 1].max()
                if mx > 0:
                    ab_data[:, 1] /= mx
            return ab_data[:, 0], ab_data[:, 1]
    raise ValueError(f"No AB or EX spectrum found for '{dye_name}' in fpbase")


def absorption_peak_nm(dye_name):
    wl, vals = _get_absorption_data(dye_name)
    return float(wl[np.argmax(vals)])


def direct_excitation_frac(dye_name, laser_nm):
    wl, vals = _get_absorption_data(dye_name)
    max_val = vals.max()
    if max_val == 0:
        return 0.0
    return float(np.interp(laser_nm, wl, vals) / max_val)


# --- absorption peaks for all dyes ---
print('Fetching absorption peaks...')
peak_wl     = {d: absorption_peak_nm(d) for d in dye_list}
sorted_dyes = sorted(dye_list, key=lambda d: peak_wl[d])

print(f'\n{"Dye":15s}  {"peak (nm)":>10s}')
print('-' * 28)
for d in sorted_dyes:
    print(f'{d:15s}  {peak_wl[d]:10.0f}')

# --- pre-compute direct excitation fractions for all dyes at all laser lines ---
exc_table = {d: {l: direct_excitation_frac(d, l) for l in LASER_WAVELENGTHS}
             for d in sorted_dyes}

print(f'\nDirect excitation (all laser lines) — ⚠ marks >{ACC_DIRECT_EXC_LIMIT*100:.0f}%:')
header = f'  {"Dye":15s}  {"peak":>6s}' + ''.join(f'  {l} nm' for l in LASER_WAVELENGTHS)
print(header)
print('  ' + '-' * (len(header) - 2))
for d in sorted_dyes:
    row = f'  {d:15s}  {peak_wl[d]:6.0f}'
    for l in LASER_WAVELENGTHS:
        f = exc_table[d][l]
        tag = '⚠ ' if f > ACC_DIRECT_EXC_LIMIT else '  '
        row += f'  {tag}{f*100:4.1f}%'
    print(row)

# --- viable laser lines per donor ---
viable_lasers_for = {}
print(f'\nViable laser lines per donor (donor abs >= {DONOR_EXC_THRESHOLD*100:.0f}% of peak):')
for d in sorted_dyes:
    vl = [l for l in LASER_WAVELENGTHS if exc_table[d][l] >= DONOR_EXC_THRESHOLD]
    viable_lasers_for[d] = vl
    print(f'  {d:15s}  {[str(l)+" nm" for l in vl]}')

# --- enumerate all valid donors and their acceptor pairs ---
all_candidates = []
print('\nValid donor → acceptor combinations:')
for donor in sorted_dyes:
    acceptors = [d for d in sorted_dyes if peak_wl[d] > peak_wl[donor]]
    if len(acceptors) < 2:
        continue
    pairs = list(combinations(acceptors, 2))
    print(f'  {donor} ({peak_wl[donor]:.0f} nm) → {len(pairs)} pair(s) '
          f'[{", ".join(acceptors)}]')
    for a1, a2 in pairs:
        all_candidates.append((donor, a1, a2))

print(f'\nTotal unique triads: {len(all_candidates)}')

# --- compute metrics; expand to one row per (triad, viable laser line) ---
# Triads with no viable laser still appear once with laser_nm=None, laser_practical=False.
print()
results = []
for donor, a1, a2 in all_candidates:
    print(f'  {donor} → {a1} + {a2} ...')
    m = triad_metrics(donor, a1, a2)
    vl = viable_lasers_for[donor]

    if not vl:
        row = dict(m)
        row['laser_nm']       = None
        row['exc_a1']         = None
        row['exc_a2']         = None
        row['laser_practical'] = False
        results.append(row)
    else:
        for l in vl:
            row = dict(m)
            row['laser_nm']        = l
            row['exc_a1']          = round(exc_table[a1][l], 4)
            row['exc_a2']          = round(exc_table[a2][l], 4)
            row['laser_practical'] = bool(
                exc_table[a1][l] <= ACC_DIRECT_EXC_LIMIT and
                exc_table[a2][l] <= ACC_DIRECT_EXC_LIMIT
            )
            results.append(row)

print(f'\nDone. {len(results)} rows ({len(all_candidates)} triads × viable laser lines).')

## Summary table

Sorted by triangle area (largest = most distinguishable). All triads are shown.  
- **practical**: False (red) if either acceptor yields < 500 photons at its own R₀  
- **A1A2_crosstalk**: True (red) if R₀(A1,A2) > 3 nm  
- **viable_lasers**: laser lines where the donor absorbs ≥ 40 % of its peak  
- **laser_practical**: False (red) if either acceptor exceeds 15 % direct excitation at any viable donor laser line — triads with False here are excluded from "best"

In [ ]:
display_cols = [
    'donor', 'acc1', 'acc2',
    'laser_nm',
    'R0_DA1', 'R0_DA2',
    'QY_D', 'QY_A1', 'QY_A2',
    'N_A1_at_R0', 'N_A2_at_R0',
    'area', 'min_dist',
    'R0_A1A2',
    'practical', 'A1A2_crosstalk',
    'exc_a1', 'exc_a2', 'laser_practical',
]

results_df = pd.DataFrame(results)[display_cols].sort_values(
    ['laser_practical', 'practical', 'area'], ascending=[False, False, False]
)


def _flag_false(v):
    return 'color: red; font-weight: bold' if v is False or v == False else ''


def _flag_true(v):
    return 'color: red; font-weight: bold' if v is True or v == True else ''


def _flag_high_exc(v):
    if v is None:
        return ''
    return 'color: red; font-weight: bold' if v > ACC_DIRECT_EXC_LIMIT else ''


fmt = {
    'R0_DA1': '{:.1f} nm', 'R0_DA2': '{:.1f} nm', 'R0_A1A2': '{:.1f} nm',
    'QY_D': '{:.2f}', 'QY_A1': '{:.2f}', 'QY_A2': '{:.2f}',
    'area': '{:.4f}', 'min_dist': '{:.4f}',
    'exc_a1': lambda v: f'{v*100:.1f}%' if v is not None else '—',
    'exc_a2': lambda v: f'{v*100:.1f}%' if v is not None else '—',
    'laser_nm': lambda v: f'{v} nm' if v is not None else '—',
}

results_df.style \
    .background_gradient(subset=['area', 'min_dist'], cmap='Greens') \
    .background_gradient(subset=['R0_A1A2'], cmap='Oranges') \
    .map(_flag_false, subset=['practical', 'laser_practical']) \
    .map(_flag_true,  subset=['A1A2_crosstalk']) \
    .map(_flag_high_exc, subset=['exc_a1', 'exc_a2']) \
    .format(fmt)

## Distance sweep — photon counts

For the top-ranked triad: vary r₁ (D–A1) with r₂ fixed at R₀(DA2), and vice versa.
Shows where each acceptor's signal drops below the shot-noise threshold.

In [ ]:
# Best row: highest area among those passing both photon-count and laser-excitation criteria
practical_rows = [m for m in results if m['practical'] and m['laser_practical']]
if not practical_rows:
    print('WARNING: no row passes all criteria — falling back to all results.')
    practical_rows = results

best = max(practical_rows, key=lambda m: m['area'])
laser_str = f'{best["laser_nm"]} nm' if best['laser_nm'] is not None else '—'
print(f'Best: {best["donor"]} → {best["acc1"]} + {best["acc2"]}  @  {laser_str}')
print(f'  area={best["area"]:.4f}, exc_a1={best["exc_a1"]}, exc_a2={best["exc_a2"]}')

R0_DA1 = best['_R0_DA1']
R0_DA2 = best['_R0_DA2']
QY_D   = best['_QY_D']
QY_A1  = best['_QY_A1']
QY_A2  = best['_QY_A2']
rgb_D  = best['_rgb_D']
rgb_A1 = best['_rgb_A1']
rgb_A2 = best['_rgb_A2']

r_sweep = np.linspace(0.3, 3.5, 200)

fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=False)

for ax, sweep_idx, fixed_r_mult, sweep_label, fixed_label in [
    (axes[0], 1, 1.0, r'$r_1$ / $R_{0,DA1}$', r'$r_2 = R_{0,DA2}$'),
    (axes[1], 2, 1.0, r'$r_2$ / $R_{0,DA2}$', r'$r_1 = R_{0,DA1}$'),
]:
    N_D_arr, N_A1_arr, N_A2_arr = [], [], []
    for mult in r_sweep:
        r1 = mult * R0_DA1 if sweep_idx == 1 else fixed_r_mult * R0_DA1
        r2 = fixed_r_mult * R0_DA2 if sweep_idx == 1 else mult * R0_DA2
        n_d, n_a1, n_a2 = triad_photons(r1, r2, R0_DA1, R0_DA2, QY_D, QY_A1, QY_A2)
        N_D_arr.append(n_d); N_A1_arr.append(n_a1); N_A2_arr.append(n_a2)

    ax.plot(r_sweep, N_D_arr,  label=f'Donor ({best["donor"]})',  color='steelblue')
    ax.plot(r_sweep, N_A1_arr, label=f'A1 ({best["acc1"]})',      color='tomato')
    ax.plot(r_sweep, N_A2_arr, label=f'A2 ({best["acc2"]})',      color='seagreen')
    ax.axhline(N_MIN, color='gray', ls='--', lw=1, label=f'Threshold ({N_MIN} ph)')
    ax.axvline(1.0,   color='gray', ls=':',  lw=1)
    ax.set_xlabel(sweep_label)
    ax.set_ylabel('Detected photons')
    ax.set_title(f'Sweep {sweep_label.split("/")[0].strip()}, fixed {fixed_label}')
    ax.legend(fontsize=8)
    ax.set_yscale('log')

fig.suptitle(f'{best["donor"]} → {best["acc1"]} + {best["acc2"]}  (excitation: {laser_str})',
             fontsize=12)
plt.tight_layout()
plt.show()

## Ternary plot — RGB trajectory over a 2D distance grid

Each point is one (r₁, r₂) combination.  Colour encodes total photon count (brighter = more photons).
The three limiting points and the distinguishability triangle are marked.  
Axes: **top = R**, **left = G**, **right = B** (PlottingFunctions convention).

In [ ]:
n_grid  = 40
r1_grid = np.linspace(0.3 * R0_DA1, 3.0 * R0_DA1, n_grid)
r2_grid = np.linspace(0.3 * R0_DA2, 3.0 * R0_DA2, n_grid)

from pathlib import Path
OUT_DIR = Path('/home/jbeckwith/Documents/pCloud/Chemistry/Lee/Data/Simulation/20260319_3WayFRET/no488')
OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f'Saving to: {OUT_DIR}')

# dye_rgb() returns [B, G, R]: index 0=B, 1=G, 2=R
# PlottingFunctions convention: ax.scatter(R, G, B) → top=R, left=G, right=B
R_pts, G_pts, B_pts, N_pts = [], [], [], []
for r1 in r1_grid:
    for r2 in r2_grid:
        rgb_norm, N_tot = triad_rgb_norm(r1, r2, R0_DA1, R0_DA2, QY_D, QY_A1, QY_A2,
                                         rgb_D, rgb_A1, rgb_A2)
        B_pts.append(rgb_norm[0])   # index 0 = B
        G_pts.append(rgb_norm[1])   # index 1 = G
        R_pts.append(rgb_norm[2])   # index 2 = R
        N_pts.append(N_tot)

R_pts = np.array(R_pts); G_pts = np.array(G_pts)
B_pts = np.array(B_pts); N_pts = np.array(N_pts)

fig = plt.figure(figsize=(6, 5))
ax  = fig.add_subplot(projection='ternary')
_ternary_axis_setup(ax)

sc = ax.scatter(R_pts, G_pts, B_pts, c=np.log10(N_pts),
                cmap='viridis', s=18, alpha=0.7, zorder=2)
cbar = fig.colorbar(sc, ax=ax, pad=0.1)
cbar.set_label('log₁₀(total photons)', fontsize=9)

# limiting points: P = [B, G, R] → scatter(R=P[2], G=P[1], B=P[0])
P_D  = best['_P_D']
P_A1 = best['_P_A1']
P_A2 = best['_P_A2']
for P, label, color in [
    (P_D,  f'D ({best["donor"]})',  'steelblue'),
    (P_A1, f'A1 ({best["acc1"]})', 'tomato'),
    (P_A2, f'A2 ({best["acc2"]})', 'seagreen'),
]:
    ax.scatter(P[2], P[1], P[0], color=color, s=80, zorder=5, label=label)

# distinguishability triangle
tri_R = [P_D[2], P_A1[2], P_A2[2], P_D[2]]
tri_G = [P_D[1], P_A1[1], P_A2[1], P_D[1]]
tri_B = [P_D[0], P_A1[0], P_A2[0], P_D[0]]
ax.plot(tri_R, tri_G, tri_B, 'k--', lw=1, zorder=4)
ax.fill(tri_R, tri_G, tri_B, color='gray', alpha=0.12, zorder=1)

ax.set_tlabel('B', fontsize=10)
ax.set_llabel('R', fontsize=10)
ax.set_rlabel('G', fontsize=10)
ax.legend(loc='upper right', fontsize=8, bbox_to_anchor=(1.35, 1.0))
laser_str = f'{best["laser_nm"]} nm' if best['laser_nm'] is not None else '—'
ax.set_title(f'{best["donor"]} → {best["acc1"]} + {best["acc2"]}  ({laser_str})\n'
             f'area = {best["area"]:.4f}, min_dist = {best["min_dist"]:.4f}',
             fontsize=10)
safe = lambda s: s.replace(' ', '_').replace('/', '-')
fname = OUT_DIR / f'BestTriad_{safe(best["donor"])}_{safe(best["acc1"])}_{safe(best["acc2"])}.svg'
plt.savefig(fname, dpi=600, format='svg')
plt.show()

## Ternary plots — all triads (saved individually)

One PNG per unique (donor, A1, A2) triad, written to the output folder below.  
Each title shows the viable laser lines with ✓ (laser_practical) or ✗ (direct excitation too high).

In [ ]:
from pathlib import Path
from collections import defaultdict

OUT_DIR = Path('/home/jbeckwith/Documents/pCloud/Chemistry/Lee/Data/Simulation/20260319_3WayFRET/no488')
OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f'Saving to: {OUT_DIR}')

# Group rows by unique triad
triad_groups = defaultdict(list)
for row in results:
    triad_groups[(row['donor'], row['acc1'], row['acc2'])].append(row)

for (donor, acc1, acc2), rows in triad_groups.items():
    m = rows[0]   # P_D/P_A1/P_A2 identical for all rows of this triad

    fig = plt.figure(figsize=(5, 4.5))
    ax  = fig.add_subplot(projection='ternary')
    _ternary_axis_setup(ax)

    P_D  = m['_P_D']   # [B, G, R]
    P_A1 = m['_P_A1']
    P_A2 = m['_P_A2']

    # PlottingFunctions convention: ax.scatter(R, G, B) → top=R, left=G, right=B
    # P[2]=R, P[1]=G, P[0]=B
    for P, label, color in [
        (P_D,  f'D ({donor})',  'steelblue'),
        (P_A1, f'A1 ({acc1})', 'tomato'),
        (P_A2, f'A2 ({acc2})', 'seagreen'),
    ]:
        ax.scatter(P[2], P[1], P[0], color=color, s=80, zorder=5, label=label)

    tri_R = [P_D[2], P_A1[2], P_A2[2], P_D[2]]
    tri_G = [P_D[1], P_A1[1], P_A2[1], P_D[1]]
    tri_B = [P_D[0], P_A1[0], P_A2[0], P_D[0]]
    ax.plot(tri_R, tri_G, tri_B, 'k--', lw=1)
    ax.fill(tri_R, tri_G, tri_B, color='gray', alpha=0.15)

    ax.set_tlabel('B', fontsize=8)
    ax.set_llabel('R', fontsize=8)
    ax.set_rlabel('G', fontsize=8)
    ax.legend(fontsize=6)

    laser_parts = [
        f'{r["laser_nm"]} nm {"✓" if r["laser_practical"] else "✗"}'
        for r in rows if r['laser_nm'] is not None
    ]
    practical_str = '✓' if m['practical'] else '✗ low QY'
    crosstalk_str = f"R₀(A1A2)={m['R0_A1A2']} nm{'  ⚠' if m['A1A2_crosstalk'] else ''}"
    ax.set_title(
        f'{donor} → {acc1} + {acc2}\n'
        f'area={m["area"]:.4f}  {practical_str}  {" | ".join(laser_parts)}\n'
        f'{crosstalk_str}',
        fontsize=8,
    )

    safe = lambda s: s.replace(' ', '_').replace('/', '-')
    fname = OUT_DIR / f'{safe(donor)}_{safe(acc1)}_{safe(acc2)}.png'
    fig.savefig(fname, dpi=150, bbox_inches='tight')
    plt.close(fig)
    print(f'  {fname.name}')

print(f'\nDone — {len(triad_groups)} files saved to {OUT_DIR}')

In [ ]:
results_df.to_csv(OUT_DIR / 'Summary_Table.csv', index=False)
print(f'Summary table saved to {OUT_DIR / "Summary_Table.csv"}')